In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
import requests
import xml.etree.ElementTree as ET
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [5]:
@tool
def search_tool(query: str) -> str:
    """Search Google News for recent articles about a topic."""

    try:
        response = requests.get(
            "https://news.google.com/rss/search",
            params={"q": query, "hl": "en-US", "gl": "US", "ceid": "US:en"},
            timeout=15,
        )
        response.raise_for_status()
        items = ET.fromstring(response.content).findall("./channel/item")[:5]
    except (requests.RequestException, ET.ParseError) as exc:
        raise RuntimeError(f"Google News search failed: {exc}") from exc

    if not items:
        return f"No Google News articles found for: {query}"

    return "\n\n".join(
        f"Title: {item.findtext('title')}\n"
        f"URL: {item.findtext('link')}\n"
        f"Published: {item.findtext('pubDate')}"
        for item in items
    )

In [6]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:

    """Get the exchange rate from the base currency to the target currency."""

    url = f"https://v6.exchangerate-api.com/v6/{os.environ['EXCHANGE_RATE_API_KEY']}/pair/{base_currency}/{target_currency}"

    response = requests.get(url)

    return response.json()


In [7]:
llm = ChatOpenAI(model = "gpt-6-luna")

In [8]:
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub
from langchain.agents import create_agent

Using create_agent

In [9]:
agent = create_agent(
    model=llm,
    tools=[search_tool,get_conversion_factor],
    system_prompt="You are a helpful assistant. Use search tool and get_conversion_factor tool when current or external information is needed."
)

In [15]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Search the country in the south of India and then find its conversion rate from inr to that currency that the country use."}
    ],
    "language": "English"
})  

In [16]:
response

{'messages': [HumanMessage(content='Search the country in the south of India and then find its conversion rate from inr to that currency that the country use.', additional_kwargs={}, response_metadata={}, id='f8e0b71c-b78f-4dfa-8798-f918c25cbf4d'),
  AIMessage(content=[{'id': 'rs_0afedc39ae2536cd006b05b96b677c87d28e5d899d0b762118', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABrBblrhSM6pKXRsMBTqka4oZx-LWJzLB1xvNgWEjiNrWZY8Nc0HksCbUyUaXEWHGPBEBl3U4XSHYR5utO9W5rAjwG0vZC6FypADcWd0QU54_UYBeIsBicUgY5JBWPOi_LHLcJmUW6nhD35jUlSSgy2LK8QYnAwEwPgOk561B4rshE3LrUbQ7BAJk0EawH-CIrqJmG1tD06ChvcRcmrhXeVC6vYw2tkDr58aLCuCsj0EG9dw3tLvGWgg7YqWFi_qNPuzSWjF3VMh-953qaJVRORn6jERO_ihj0o9R6lKdNcy7K4T-aDrz3pTIsWCGCtKt9MBS18WirGBW3APS2e6d9ZNVzgVkrwOopS4ccJxnO-Oct6znSTf-hoUKfCbDdqM7TV7-WvEmQhpATcsUk9XUi6Agc4xQ0lK_78LENqa95XF6oV3r3gArm2hoJ_8mrHZJHf97gA6X-z7dwAU4uwySRHCg_r8FqBm-0v3WiaYh1COODy0G7Mr5mymXT7J6_Ky0S8ekJmpk60cDlSGnAZMdX6AMJA12cUDrFp0wqHdLlw-1FMya-mF-I2FbQ5XAoqjocFJpezEB9O8

In [17]:
print(response["messages"][-1].content[-1]["text"])

The country south of India is **Sri Lanka**, and its currency is the **Sri Lankan rupee (LKR)**. The current rate returned is **₹1 INR = 3.5817 LKR** (updated November 24, 2026). Rates may change.
